## 기본 실행 코드
### 이것들은 실행시켜야 어느 단계든 원활하게 돌아감

In [ ]:
# 코랩을 시작할 때 아래코드를 한 번 돌려줍니다.
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install webdriver_manager

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-chromedriver is already the newest version (1:85.0.4183.83-0ubuntu2.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 17 n

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import sys
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

In [ ]:
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from openpyxl import Workbook
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
import time
import datetime
import requests

In [ ]:
service = Service(executable_path=r'chromedriver')
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(options=options)

In [ ]:
# url
url = 'https://m.place.naver.com/restaurant/1085956231/review/visitor?entry=ple&reviewSort=recent'

# BS4 setting for secondary access
session = requests.Session()
headers = {
    "User-Agent": "user value"}

retries = Retry(total=5,
                backoff_factor=0.1,
                status_forcelist=[500, 502, 503, 504])

session.mount('http://', HTTPAdapter(max_retries=retries))

In [ ]:
# New xlsx file
now = datetime.datetime.now()
xlsx = Workbook()
list_sheet = xlsx.create_sheet('output')
list_sheet.append(['nickname', 'content', 'date', 'revisit'])

In [ ]:
# Start crawling/scraping!
try:
    res = driver.get(url)
    driver.implicitly_wait(30)

    # Pagedown
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)

    try:
        while True:
            driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[7]/div[2]/div[3]/div[2]').click()
            time.sleep(0.4)
    except Exception as e:
        print('finish')

    time.sleep(25)
    html = driver.page_source
    bs = BeautifulSoup(html, 'lxml')
    reviews = bs.select('li.YeINN')

    for r in reviews:
        nickname = r.select_one('div.sBWyy')
        content = r.select_one('div.ZZ4OK.IwhtZ')
        date = r.select('span.P1zUJ>span.place_blind')[1]
        revisit = r.select('div.sb8UA>span.P1zUJ')[1]

        # exception handling
        nickname = nickname.text if nickname else ''
        content = content.text if content else ''
        date = date.text if date else ''
        revisit = revisit.text if revisit else ''
        time.sleep(0.06)

        print(nickname, '/', content, '/', date, '/', revisit)
        list_sheet.append([nickname, content, date, revisit])
        time.sleep(0.06)
    # Save the file
    file_name = '/data/raw/naver_review_' + now.strftime('%Y-%m-%d_%H-%M-%S') + '.xlsx'
    xlsx.save(file_name)

except Exception as e:
    print(e)
    # Save the file(temp)
    file_name = 'naver_review_' + now.strftime('%Y-%m-%d_%H-%M-%S') + '.xlsx'
    xlsx.save(file_name)

finish
list index out of range


## 연극 월간 랭킹 가져오기

In [ ]:
driver.maximize_window()
driver.get("http://ticket.interpark.com/TPGoodsList.asp?Ca=Dra&Sort=3")
time.sleep(3)
driver.implicitly_wait(1)
time.sleep(3)
text = []
place = []
date = []
entire = []

rows=driver.find_elements(By.CLASS_NAME, 'fw_bold')
for i, row in enumerate(rows):
    text.append(row.text)

play_place=driver.find_elements(By.CLASS_NAME, 'Rkdate')
for i, p in enumerate(play_place):
  entire.append(p.text)

place = entire[0::2]
date = entire[1::2]

In [ ]:
import pandas as pd
now = datetime.datetime.now()
play_rank = pd.DataFrame({"연극 이름":text, "공연장": place, "기간": date})
play_rank.to_csv("/data/raw/play_rank" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

In [ ]:
driver.maximize_window()
driver.get("http://ticket.interpark.com/TPGoodsList.asp?Ca=Mus&Sort=3")
time.sleep(3)
driver.implicitly_wait(1)
time.sleep(3)
text = []
place = []
date = []
entire = []

rows=driver.find_elements(By.CLASS_NAME, 'fw_bold')
for i, row in enumerate(rows):
    text.append(row.text)

play_place=driver.find_elements(By.CLASS_NAME, 'Rkdate')
for i, p in enumerate(play_place):
  entire.append(p.text)

place = entire[0::2]
date = entire[1::2]

In [ ]:
import pandas as pd
now = datetime.datetime.now()
play_rank = pd.DataFrame({"뮤지컬 이름":text, "공연장": place, "기간": date})
play_rank.to_csv("/data/raw/musical_rank" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

## 크롤링 된 데이터 정제

In [ ]:
import pandas as pd
play_rank = pd.read_csv('/data/raw/play_rank_sample.csv', encoding='utf-8')
play_rank = pd.DataFrame(play_rank[:156])
play_rank

,Unnamed: 0,연극 이름,공연장,기간
0,0,연극 너의 목소리가 들려,도토리씨어터,2023.04.03~\n2023.10.31
1,1,대학로 1위 연극 〈쉬어매드니스〉,콘텐츠박스(KONTENTZ BOX),2015.11.12~\n2023.10.31
2,2,행오버,정극장,2023.05.01~\n2023.10.31
3,3,연극 〈바닷마을 다이어리〉,예술의전당 자유소극장,2023.10.08~\n2023.11.19
4,4,연극 한뼘사이,라온아트홀,2017.03.01~\n2023.10.31
...,...,...,...,...
151,151,섹시코미디연극 나의PS파트너 (청주),소명아트홀,2023.12.01~\n2023.12.31
152,152,쇼팔다 미친 유령 극단,수유연극실험실,2023.09.01~\n2023.10.15
153,153,스쁘라브카-열람,예술공간 혜화,2023.11.25~\n2023.11.26
154,154,스위치 - 대전,작은극장 다함,2023.09.22~\n2024.01.01


In [ ]:
musical_rank = pd.read_csv('/data/raw/musical_rank_sample.csv', encoding='utf-8')
musical_rank = pd.DataFrame(musical_rank[:])
musical_rank

,Unnamed: 0,뮤지컬 이름,공연장,기간
0,0,뮤지컬 〈레베카〉 10주년 기념공연,블루스퀘어 신한카드홀,2023.08.19~\n2023.11.19
1,1,태양의서커스 〈루치아〉,잠실종합운동장 내 빅탑,2023.10.25~\n2023.12.17
2,2,뮤지컬 〈오페라의 유령〉 - 서울,샤롯데씨어터,2023.07.21~\n2023.11.19
3,3,뮤지컬 ＇벤허＇,LG아트센터 서울 LG SIGNATURE 홀,2023.09.02~\n2023.11.19
4,4,뮤지컬 〈멤피스〉,충무아트센터 대극장,2023.07.20~\n2023.10.22
...,...,...,...,...
151,151,하트시그널,대학로 하마씨어터(구 가든씨어터),2022.11.02~\n2023.11.30
152,152,한국문학뮤지컬 시리즈 I ‘메밀꽃필무렵’ - 대전,대전시립연정국악원 큰마당,2023.10.13~\n2023.10.14
153,153,K뮤지컬 〈정수정전〉 - 공주,공주문예회관 대공연장,2023.10.20~\n2023.10.20
154,154,PIUM Project 뮤지컬 갈라 콘서트 - 대전,대전예술의전당 아트홀,2023.10.12~\n2023.10.12


In [ ]:
play_string = play_rank['연극 이름'].apply(str)
musical_string = musical_rank['뮤지컬 이름'].apply(str)

In [ ]:
list = [37, 67, 70, 89, 131, 140]
for j in list:
  test_str = play_string[j]
  test_str = test_str.replace("［", "〈")
  test_str = test_str.replace("］", "〉")
  print(test_str)
  play_string[j] = test_str
#play_string[198] = m

공포연극 〈자취〉
〈연애하기 좋은 날 : 당근거래〉 - 전주
〈청소년극〉〈Tank ; 0-24〉
2023 제작연극 헉슬리〈멋진 신세계〉-대전
로맨틱연극〈연애하기좋은날:당근거래〉
문화가 있는 날 Ⅳ NT Live〈리어왕〉- 대전


In [ ]:
list = [24, 140, 141]
for j in list:
  test_str = musical_string[j]
  test_str = test_str.replace("［", "〈")
  test_str = test_str.replace("］", "〉")
  print(test_str)
  musical_string[j] = test_str
test_str = musical_string[130]
test_str = test_str.replace("「", "<")
test_str = test_str.replace("」", "<")
musical_string[130] = test_str

musical_string

이은결 〈더 일루션 - 마스터피스〉
이은결〈더 일루션 - 마스터피스〉- 고양
이은결〈더 일루션 - 마스터피스〉- 대구


0               뮤지컬 〈레베카〉 10주년 기념공연
1                      태양의서커스 〈루치아〉
2                뮤지컬 〈오페라의 유령〉 - 서울
3                          뮤지컬 ＇벤허＇
4                         뮤지컬 〈멤피스〉
                   ...             
151                           하트시그널
152     한국문학뮤지컬 시리즈 I ‘메밀꽃필무렵’ - 대전
153                K뮤지컬 〈정수정전〉 - 공주
154    PIUM Project 뮤지컬 갈라 콘서트 - 대전
155                   The Hook - 성남
Name: 뮤지컬 이름, Length: 156, dtype: object

In [ ]:
play_string = play_string.str.replace(pat=r"［", repl=r'(', regex=True)
play_string = play_string.str.replace(pat=r"］", repl=r')', regex=True)
play_string = play_string.str.replace(pat=r'\([^)]*\)', repl="", regex=True)
play_string = play_string.str.replace(pat="연극", repl="")
#play_string = play_string.str.replace(pat=r"-")

In [ ]:
musical_string = musical_string.str.replace(pat=r"［", repl=r'(', regex=True)
musical_string = musical_string.str.replace(pat=r"］", repl=r')', regex=True)
musical_string = musical_string.str.replace(pat=r'\([^)]*\)', repl="", regex=True)
musical_string = musical_string.str.replace(pat="뮤지컬", repl="")

In [ ]:
i = 0
while(i<155):
  if(play_string.str.contains('-')[i]):
    play_string[i] = play_string[i].split("-")[0]
  if(play_string.str.contains("－")[i]):
    play_string[i] = play_string[i].split("－")[0]
  i += 1

In [ ]:
i = 0
while(i<155):
  if(musical_string.str.contains('-')[i]):
    musical_string[i] = musical_string[i].split("-")[0]
  if(musical_string.str.contains("－")[i]):
    musical_string[i] = musical_string[i].split("－")[0]
  i += 1

In [ ]:
import re
i = 0
while(i < 155):
  if(play_string.str.contains('〈')[i]):
    test_str = play_string[i]
    p = re.compile('〈[^)]+〉')
    try:
      m = p.findall(test_str)[0]
      play_string[i] = m
    except:
      i += 1
      continue
  i += 1

In [ ]:
import re
i = 0
while(i < 155):
  if(musical_string.str.contains('〈')[i]):
    test_str = musical_string[i]
    p = re.compile('〈[^)]+〉')
    try:
      m = p.findall(test_str)[0]
      musical_string[i] = m
    except:
      i += 1
      continue
  i += 1

In [ ]:
play_string = play_string.str.replace(pat=r"〈", repl=r'', regex=True)
play_string = play_string.str.replace(pat=r"〉", repl=r'', regex=True)
#play_string = play_string.str.replace(pat=r" ", repl=r'', regex=True)

In [ ]:
musical_string = musical_string.str.replace(pat=r"〈", repl=r'', regex=True)
musical_string = musical_string.str.replace(pat=r"〉", repl=r'', regex=True)
#musical_string = musical_string.str.replace(pat=r" ", repl=r'', regex=True)

In [ ]:
play_list = pd.DataFrame(play_string)
musical_list = pd.DataFrame(musical_string)

In [ ]:
now = datetime.datetime.now()
play_list.to_csv("/data/raw/play_refine" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')
musical_list.to_csv("/data/raw/musical_refine" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

## play_rank, musical_rank 읽기

In [ ]:
import pandas as pd
play_refine= pd.read_csv('/data/raw/play_refine2023-10-12.csv', encoding='utf-8')
play_refine = pd.DataFrame(play_refine['연극 이름'])
play_refine

,연극 이름
0,너의 목소리가 들려
1,쉬어매드니스
2,행오버
3,바닷마을 다이어리
4,한뼘사이
...,...
151,섹시코미디 나의PS파트너
152,쇼팔다 미친 유령 극단
153,스쁘라브카
154,스위치


In [ ]:
musical_refine = pd.read_csv('/data/raw/musical_refine2023-10-12.csv', encoding='utf-8')
musical_refine = pd.DataFrame(musical_refine['뮤지컬 이름'])
musical_refine

,뮤지컬 이름
0,레베카
1,루치아
2,오페라의 유령
3,＇벤허＇
4,멤피스
...,...
151,하트시그널
152,한국문학 시리즈 I ‘메밀꽃필무렵’
153,정수정전
154,PIUM Project 갈라 콘서트


In [ ]:
play_rank = pd.read_csv('/data/raw/play_rank2023-10-09.csv', encoding='utf-8')
play_rank = pd.DataFrame(data=play_rank[:156], columns=['공연장', '기간'])
play_rank

,공연장,기간
0,도토리씨어터,2023.04.03~\n2023.10.31
1,콘텐츠박스(KONTENTZ BOX),2015.11.12~\n2023.10.31
2,정극장,2023.05.01~\n2023.10.31
3,예술의전당 자유소극장,2023.10.08~\n2023.11.19
4,라온아트홀,2017.03.01~\n2023.10.31
...,...,...
151,소명아트홀,2023.12.01~\n2023.12.31
152,수유연극실험실,2023.09.01~\n2023.10.15
153,예술공간 혜화,2023.11.25~\n2023.11.26
154,작은극장 다함,2023.09.22~\n2024.01.01


In [ ]:
musical_rank = pd.read_csv('/data/raw/musical_rank2023-10-09.csv', encoding='utf-8')
musical_rank = pd.DataFrame(data=musical_rank[:156], columns=['공연장', '기간'])
musical_rank

,공연장,기간
0,블루스퀘어 신한카드홀,2023.08.19~\n2023.11.19
1,잠실종합운동장 내 빅탑,2023.10.25~\n2023.12.17
2,샤롯데씨어터,2023.07.21~\n2023.11.19
3,LG아트센터 서울 LG SIGNATURE 홀,2023.09.02~\n2023.11.19
4,충무아트센터 대극장,2023.07.20~\n2023.10.22
...,...,...
151,대학로 하마씨어터(구 가든씨어터),2022.11.02~\n2023.11.30
152,대전시립연정국악원 큰마당,2023.10.13~\n2023.10.14
153,공주문예회관 대공연장,2023.10.20~\n2023.10.20
154,대전예술의전당 아트홀,2023.10.12~\n2023.10.12


In [ ]:
play_DF = pd.concat([play_refine, play_rank], axis=1)
musical_DF = pd.concat([musical_refine, musical_rank], axis=1)

## playDB 크롤링

In [ ]:
driver.get("http://www.playdb.co.kr/Index.asp")
time.sleep(3)
driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(play_DF['연극 이름'][i])
driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(Keys.ENTER)
time.sleep(3)
info_time = play_DF['기간'][i].split("~")[0]
info_time=datetime.strptime(info_time,'%Y.%m.%d')
comp_time = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table[1]/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[1]').text
comp_time=comp_time.split(" ")[0]
comp_time = datetime.strptime(comp_time,'%Y/%m/%d')

In [ ]:
def playDB(DF, ran, i):
  if(ran == 0):
    print("Search None");
  elif(ran == 1):
    driver.find_element(By.XPATH, '//*[@id="dvSpecial"]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[3]/table/tbody/tr[1]/td/table/tbody/tr[2]/td/b/a')
  else:
    for j in range(1, int(ran)):
      place = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[2]/a').text
      if(DF['공연장'][i] == place):
          driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
          time.sleep(3)
          driver.find_element(By.XPATH, '//*[@id="Tab2"]').click()
          try:
            print(driver.find_element(By.XPATH, '/html/body/table/tbody/tr[2]/td/table/tbody/tr[3]/td').text)
            break
          except:
            print("Explain None")


In [ ]:
def info_news(driver):
  try:
    time.sleep(3)
    driver.find_element(By.XPATH, '//*[@id="contents"]/div[1]/div[1]/ul[1]/li[2]/a').click()
    iframe_element = driver.find_element(By.ID, "iFrmContent")  # iframe의 id를 사용하여 찾는 예시
    driver.switch_to.frame(iframe_element)
    text = driver.find_element(By.XPATH, '/html/body/table/tbody/tr[2]/td/table/tbody/tr[3]/td').text
  except:
    text = ""
  return text

In [ ]:
play_DF['연극 이름'][i]

'너의목소리가들려'

In [ ]:
import datetime
now = datetime.datetime.now()

In [ ]:
import re
from datetime import datetime, timedelta
play_info = []
for i in range(156):
  driver.get("http://www.playdb.co.kr/Index.asp")
  time.sleep(3)
  driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(play_DF['연극 이름'][i])
  driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(Keys.ENTER)
  time.sleep(3)

  try:
    ran = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[1]/td/table/tbody/tr/td[2]/b').text
    ran = re.sub(r'[^0-9]', '', ran)
  except:
    try:
      driver.find_element(By.XPATH, '//*[@id="wrap"]/table/tbody/tr[3]/td/table/tbody/tr/td[1]/table[2]/tbody/tr[2]/td/table/tbody/tr/td/table/tbody/tr[2]/td/div/b')
      ran = 0
    except:
      ran = 2

  if(ran == 0):
    print("Search None");
    text = "None"
  elif(ran == 2):
    driver.find_element(By.XPATH, '//*[@id="dvSpecial"]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[3]/table/tbody/tr[1]/td/table/tbody/tr[2]/td/b/a').click()
    text = info_news(driver)
    print(text)
  else:
    info_time = play_DF['기간'][i].split("~")[0]
    info_time=datetime.strptime(info_time,'%Y.%m.%d')
    for j in range(1, int(ran)):
      comp_time = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[1]').text
      comp_time=comp_time.split(" ")[0]
      comp_time = datetime.strptime(comp_time,'%Y/%m/%d')
      place = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table[' +str(j) + ']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[2]/a').text
      if(len(place) >= len(play_DF['공연장'][i])):
        if((place.find(play_DF['공연장'][i]) >= 0) or (comp_time == info_time)):
          driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
          time.sleep(3)
          text = info_news(driver)
          print(text)
          break
      else:
        if((play_DF['공연장'][i].find(place) >= 0) or (comp_time == info_time)):
          driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
          time.sleep(3)
          text = info_news(driver)
          print(text)
          break
      if(comp_time < info_time):
        text = "None"
        print("Search None")
        break
  play_info.append(text)
play_info_DF = pd.DataFrame(play_info)
play_DF = pd.concat([play_refine, play_rank, play_info_DF], axis=1)
play_DF.to_csv("/data/raw/play_info_DF_new" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')



 
모든 일은 파라다이스 호텔 506호에서 시작된다.
남편(철수)의 외도를 알게 된 지연(부인)은 이벤트 업체를 이용해서 남편에게 사과를 받고 싶어한다.
이벤트 업체 대표 태민은 지연의 계획대로 철수에게 사과를 받아보려고 했지만,
만취한 철수는 지연을 506호에서 살해한다.
당황한 이벤트 업체 대표 태민은 철수를 급하게 507호로 옮기고 4시간만 대실한다.
술이 깬 철수에게 지난 밤 일을 설명하지만, 철수는 기억나지 않는다.
506호 살인 사건을 조사 중인 경찰이 507호를 방문하지만,
태민의 임기응변으로 위기를 넘긴다.
그런데, 507호에는 다른 2명이 더 있다.
초능력이 있다고 주장하는 508호 케이(남)
자살하기 위해 호텔을 찾은 509호 엠마(여)
케이와 엠마 두명 역시 술을 먹고 깨어보니 507호라는 황당한 말을 전한다.
507호에 만난 태민, 철수, 케이, 엠마!
만약, 경찰이 507호로 들이닥치면 4명 모두 공범이 되는 상황.
이런 말도 안되는 상황을 믿어 줄 경찰은 없다.
4명은 서로를 의심하며 서로에게 자백을 강요하기 시작하는데...
이들 4명에게 주어진 시간은 대실 4시간!
4시간 안에 모든 것을 밝혀내야 되는 상황에서
4명의 비밀이 하나씩 드러난다.
 
'서로에게 진정한 가족이 되어가는 자매들의 성장 이야기'
연극 <바닷마을 다이어리>는 일본의 영화인 ‘바닷마을 다이어리(海街 Diary)’를 원작으로 한 작품이다.
2015년 칸 영화제 경쟁부문에 초청되고 많은 일본의 영화제에 수상 이력이 있는 만큼 탄탄한 스토리를 가지고 있으며,
자극적이지는 않지만 깊은 위로를 주는 작품이라는 극찬을 받았다.
‘바닷마을 다이어리’ 한국 초연을 위해 연극 각색에 능통한 황정은 작가와 다수의 연극 무대화 경험이 있는 이준우 연출이 개발 과정에 참여를 하면서
아버지와 어머니의 부재, 연인과의 헤어짐, 소중한 사람의 죽음 등으로
어딘가 결여 되어있던 네 자매들의 마음이 서로를 통해 채워지고 위로를 받으며
진정한 가족이 되어가는 모습을 연극에서도 잘 표현할 예정

NameError: ignored

In [ ]:
play_info_DF = pd.DataFrame(play_info)
play_DF = pd.concat([play_refine, play_rank, play_info_DF], axis=1)
play_DF.to_csv("/data/raw/play_info_DF" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

In [ ]:
import re
from datetime import datetime, timedelta
musical_info = []
for i in range(156):
  driver.get("http://www.playdb.co.kr/Index.asp")
  time.sleep(3)
  driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(musical_DF['뮤지컬 이름'][i])
  driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(Keys.ENTER)
  time.sleep(3)

  try:
    ran = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[1]/td/table/tbody/tr/td[2]/b').text
    ran = re.sub(r'[^0-9]', '', ran)
  except:
    try:
      driver.find_element(By.XPATH, '//*[@id="wrap"]/table/tbody/tr[3]/td/table/tbody/tr/td[1]/table[2]/tbody/tr[2]/td/table/tbody/tr/td/table/tbody/tr[2]/td/div/b')
      ran = 0
    except:
      ran = 2

  if(ran == 0):
    print("Search None");
    text = "None"
  elif(ran == 2):
    driver.find_element(By.XPATH, '//*[@id="dvSpecial"]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[3]/table/tbody/tr[1]/td/table/tbody/tr[2]/td/b/a').click()
    text = info_news(driver)
    print(text)
  else:
    info_time = musical_DF['기간'][i].split("~")[0]
    info_time=datetime.strptime(info_time,'%Y.%m.%d')
    driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[1]/td/table/tbody/tr/td[3]/div/a').click()
    time.sleep(3)
    for j in range(1, int(ran)):
      comp_time = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[1]').text
      comp_time=comp_time.split(" ")[0]
      comp_time = datetime.strptime(comp_time,'%Y/%m/%d')
      place = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table[' +str(j) + ']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[2]/a').text
      if(len(place) >= len(musical_DF['공연장'][i])):
        if((place.find(musical_DF['공연장'][i]) >= 0) or (comp_time == info_time)):
          driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
          time.sleep(3)
          text = info_news(driver)
          print(text)
          break
      else:
        if((musical_DF['공연장'][i].find(place) >= 0) or (comp_time == info_time)):
          driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
          time.sleep(3)
          text = info_news(driver)
          print(text)
          break
      if(comp_time < info_time):
        text = "None"
        print("Search None")
        break
  musical_info.append(text)
musical_info_DF = pd.DataFrame(musical_info)
musical_DF = pd.concat([musical_refine, musical_rank, musical_info_DF], axis=1)
musical_DF.to_csv("/data/raw/musical_info_DF_new" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

더 이상 수식어가 필요 없는 가장 완벽한 뮤지컬!

역사적 10주년 기념공연으로 돌아온 대체불가 ‘메가 스테디셀러’
약 13개국, 11개 언어로 공연되며 전 세계 2,100만 관객을 전율케 한 초대형 흥행작!
2013년 한국 초연 이후, 2022년 여섯 번째 시즌까지 누적 관람객 95만명 기록!

스릴러의 거장 히치콕의 영화로도 유명한 <레베카>!
감동적인 로맨스, 반전을 거듭하는 서스펜스, 강력한 킬링 넘버!
유럽 뮤지컬 레전드! 극작가 미하엘 쿤체(Michael Kunze)와 작곡가 실베스터 르베이(Sylvester Levay)의 ‘마스터 피스’

영국의 대표 작가 대프니 듀 모리에의 소설을 원작으로,
스릴러의 거장 알프레도 히치콕의 동명의 영화로도 유명한 뮤지컬 <레베카>!
원작을 뛰어넘는 뮤지컬의 탄생이라 불리며 전세계에서 사랑받아온 대작!
숨막히도록 아름다운 대저택의 비밀이 밝혀진다!

대한민국 초연 10주년 기념 무대를 꾸밀 대한민국 최정상 스태프와 압도적인 완벽 캐스팅 라인업!

총괄프로듀서 엄홍현, 연출 로버트 요한슨, 음악감독 김문정 등 국내외 최고의 스태프 총망라!
류정한, 민영기, 에녹, 신영숙, 옥주현, 리사, 장은아, 김보경, 이지혜 등 지난 시즌의 히어로들과
테이, 이지수, 웬디 등 뉴 캐스팅으로 한층 업그레이드된, 단 한 순간에 관객을 매혹시킬 <레베카> 10주년 기념 공연!

13년 간의 긴 기다림…
마침내 한국의 유령이 온다!

환영과도 같은 무대
사라지지 않을 영원한 당신의 첫 감동!
13년 만에 오페라 하우스의 문이 열린다
대한민국 창작 뮤지컬의 압도적 수작의 귀환!
제2회 한국뮤지컬어워즈 대상, 무대예술상, 앙상블상 수상 및 11개 부문 노미네이트!
제6회 에그린뮤지컬어워드 앙상블상 수상!
작품성과 대중성을 모두 만족시킨 대작!
한국 창작 뮤지컬의 새로운 기준을 정립한 수작이 돌아온다!

월드클래스 창작 뮤지컬 명가 EMK의 제작 노하우가 더해져 새롭게 탄생할 <벤허>!
웅장한 경기장에서 펼쳐지는 마지막 전차 경주!
파노라마

In [ ]:
musical_info.count("None") + musical_info.count("")

115

In [ ]:
musical_DF.to_csv("/data/raw/musical_info_DF" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

In [ ]:
len(musical_info)

156

## 테스트 코드

In [ ]:
info_time = play_DF['기간'][i].split("~")[0]
info_time=datetime.strptime(info_time,'%Y.%m.%d')
print(info_time)

2023-04-03 00:00:00


In [ ]:
play_info.count("None") + play_info.count("")

139

In [ ]:
play_INFO = pd.DataFrame({'공연소개':play_info})
play_INFO[play_INFO['공연소개']=='None'].index

Int64Index([12, 14, 18, 20, 23, 29], dtype='int64')

In [ ]:
search_none = pd.concat([play_refine, play_rank, play_INFO], axis=1)

In [ ]:
search_none[search_none['공연소개']=='']

,연극 이름,공연장,기간,공연소개
0,너의목소리가들려,도토리씨어터,2023.04.03~\n2023.10.31,
1,쉬어매드니스,콘텐츠박스(KONTENTZ BOX),2015.11.12~\n2023.10.31,
4,한뼘사이,라온아트홀,2017.03.01~\n2023.10.31,
6,불편한편의점,대학로 스타시티 7층 후암씨어터,2023.04.08~\n2023.12.31,
7,2호선세입자,대학로 바탕골 소극장,2022.01.14~\n2023.10.31,
9,좀비오마이갓,대학로 오마이갓 전용관,2023.04.11~\n2023.10.31,
10,핫식스,도토리씨어터,2023.04.03~\n2023.10.31,
11,라면,대학로 해피씨어터,2021.01.16~\n2023.10.31,
13,죽여주는이야기,지인시어터(구.알과핵소극장),2023.07.01~\n2023.10.31,
17,운빨로맨스,컬쳐씨어터,2021.05.01~\n2023.10.31,


In [ ]:
play_info_DF = pd.DataFrame(play_info)
play_DF = pd.concat([play_refine, play_rank, play_info_DF], axis=1)
play_DF.to_csv("/data/raw/play_info_DF" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')


In [ ]:
if(comp_time < info_time):
  text = ""
  print("Search None")

Yes


In [ ]:
info_time

datetime.datetime(2023, 11, 10, 0, 0)

In [ ]:
driver.get("http://www.playdb.co.kr/Index.asp")
time.sleep(3)
driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(play_DF['연극 이름'][2])
driver.find_element(By.XPATH, '//*[@id="HDQuery"]').send_keys(Keys.ENTER)
time.sleep(3)
try:
  ran = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[1]/td/table/tbody/tr/td[2]/b').text
  ran = re.sub(r'[^0-9]', '', ran)
except:
  ran = 2
for j in range(1, int(ran)):
  comp_time = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[1]').text
  comp_time=comp_time.split(" ")[0]
  comp_time = datetime.strptime(comp_time,'%Y/%m/%d')
  place = driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table[' +str(j) + ']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[2]/td[2]/a').text
  print(place)
  if(len(place) >= len(play_DF['공연장'][2])):
    if((place.find(play_DF['공연장'][2]) >= 0) or (comp_time == info_time)):
      driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
      time.sleep(3)
      driver.find_element(By.XPATH, '//*[@id="Tab2"]').click()
      try:
        print(driver.find_element(By.XPATH, '//*[@id="wrap"]/div[3]/div[1]/div[1]/table/tbody/tr[1]/td/span').text)
        print(driver.find_element(By.XPATH, '/html/body/table/tbody/tr[2]/td/table/tbody/tr[3]/td').text)
      except:
        print("Explain None")
      break
  else:
    if((play_DF['공연장'][2].find(place) >= 0) or (comp_time == info_time)):
      driver.find_element(By.XPATH, '//*[@id="tblPlay"]/tbody/tr[1]/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td/table/tbody/tr[2]/td/table['+str(j)+']/tbody/tr[1]/td/table/tbody/tr/td[3]/table/tbody/tr[2]/td/table/tbody/tr[1]/td[1]/b/a').click()
      time.sleep(3)
      driver.find_element(By.XPATH, '//*[@id="Tab2"]').click()
      try:
        print(driver.find_element(By.XPATH, '//*[@id="wrap"]/div[3]/div[1]/div[1]/table/tbody/tr[1]/td/span').text)
        print(driver.find_element(By.XPATH, '/html/body/table/tbody/tr[2]/td/table/tbody/tr[3]/td').text)
      except:
        print("Explain None")
      break

정극장 (구 M시어터)
행오버
Explain None


In [ ]:
import requests
from bs4 import BeautifulSoup
driver.get("http://www.playdb.co.kr/playdb/PlaydbDetail.asp?sReqPlayNo=183033")

time.sleep(3)
driver.find_element(By.XPATH, '//*[@id="contents"]/div[1]/div[1]/ul[1]/li[2]/a').click()
iframe_element = driver.find_element(By.ID, "iFrmContent")  # iframe의 id를 사용하여 찾는 예시
driver.switch_to.frame(iframe_element)
driver.find_element(By.XPATH, '/html/body/table/tbody/tr[2]/td/table/tbody/tr[3]/td').text


' \n모든 일은 파라다이스 호텔 506호에서 시작된다.\n남편(철수)의 외도를 알게 된 지연(부인)은 이벤트 업체를 이용해서 남편에게 사과를 받고 싶어한다.\n이벤트 업체 대표 태민은 지연의 계획대로 철수에게 사과를 받아보려고 했지만,\n만취한 철수는 지연을 506호에서 살해한다.\n당황한 이벤트 업체 대표 태민은 철수를 급하게 507호로 옮기고 4시간만 대실한다.\n술이 깬 철수에게 지난 밤 일을 설명하지만, 철수는 기억나지 않는다.\n506호 살인 사건을 조사 중인 경찰이 507호를 방문하지만,\n태민의 임기응변으로 위기를 넘긴다.\n그런데, 507호에는 다른 2명이 더 있다.\n초능력이 있다고 주장하는 508호 케이(남)\n자살하기 위해 호텔을 찾은 509호 엠마(여)\n케이와 엠마 두명 역시 술을 먹고 깨어보니 507호라는 황당한 말을 전한다.\n507호에 만난 태민, 철수, 케이, 엠마!\n만약, 경찰이 507호로 들이닥치면 4명 모두 공범이 되는 상황.\n이런 말도 안되는 상황을 믿어 줄 경찰은 없다.\n4명은 서로를 의심하며 서로에게 자백을 강요하기 시작하는데...\n이들 4명에게 주어진 시간은 대실 4시간!\n4시간 안에 모든 것을 밝혀내야 되는 상황에서\n4명의 비밀이 하나씩 드러난다.\n '

In [ ]:
i, j, ran

(0, 1, '3')

## PlayDB 기본 정보(줄거리) 크롤링

## 배우 100명 크롤링

In [ ]:
## 연극배우
driver.get("http://www.playdb.co.kr/artistdb/list.asp?code=013004")
time.sleep(3)
driver.refresh()
driver.find_element(By.XPATH, '//*[@id="contents"]/div[2]/table/tbody/tr[1]/td/table/tbody/tr/td[2]/a[2]').click()
driver.find_element(By.XPATH, '//*[@id="contents"]/div[2]/table/tbody/tr[1]/td/table/tbody/tr/td[2]/a[1]').click()
time.sleep(2)
iframe_element = driver.find_element(By.ID, "list_ifm")  # iframe의 id를 사용하여 찾는 예시
driver.switch_to.frame(iframe_element)
play_actor = []
for i in range(1, 6):
  for j in range(3, 80, 4):
    # play_DF 에 있는 장소 추출
    name = driver.find_element(By.XPATH, '/html/body/table/tbody/tr['+str(j)+']/td/table/tbody/tr/td[1]/table/tbody/tr/td[2]/a').text
    name_work = driver.find_element(By.XPATH, '/html/body/table/tbody/tr['+str(j)+']/td/table/tbody/tr/td[5]').text
    play_actor.append([name, name_work])
    print(name, name_work)
  driver.find_element(By.XPATH, '/html/body/table/tbody/tr[84]/td/a['+str(i)+']').click()

윤은채 은밀하게 위대하게(2017)
챠이카(2016)
레베카(2016)
유희준 시간을 파는 상점(2017)
하지혜 둥근 해가 떴습니다(2012)
연극 이(爾)(2010)
짐(2007)
백시연 나도 이제 결혼하고 싶다(2021)
제5회 한국여성극작가전 - 그 집(2019)
연애플레이리스트(2018)
고형우 애니 모어 스토리(2020)
두드려라 맥베스(2018)
이인규 수상한 흥신소(2021)
라면(2021)
러브 액츄얼리(2019)
최정원 컴프롬어웨이(2023)
맘마미아 - 천안(2023)
맘마미아 - 수원(2023)
유아름 1948 여수 - 여수(2022)
1948 여수 - 여수(2020)
그해 분장실(2020)
신성록 드라큘라(2023)
벤허(2023)
스위니토드(2022)
김성진 사명(2021)
숙희(2021)
언플러그드(2020)
김화영 고목(2023)
장녀들(2023)
스카프와 나이프(2022)
이란희 노르망디(2023)
칸사이 주먹(2022)
노르망디(2022)
홍경인 준생(2023)
왓 이프(2023)
행복(2023)
홍나현 22년 2개월(2023)
붉은머리 안(2023)
홍련(2023)
남윤호 코리올라누스(2021)
보도지침(2017)
월요쇼케이스 - 보도지침(2017)
강보라 리미트(2018)
뷰티풀 라이프(2018)
변서윤 룸넘버13 - 울산(2018)
룸넘버13(2016)
내 깡패같은 로맨스(2015)
김지현 그날들 - 대구(2023)
그날들(2023)
스위니토드(2022)
이시원 극적인 하룻밤(2023)
오싹한 연애(2020)
당신 안녕(2016)
엄기준 그날들 - 수원(2023)
그날들 - 대구(2023)
그날들(2023)
윤은채 은밀하게 위대하게(2017)
챠이카(2016)
레베카(2016)
유희준 시간을 파는 상점(2017)
하지혜 둥근 해가 떴습니다(2012)
연극 이(爾)(2010)
짐(2007)
백시연 나도 이제 결혼하고 싶다(2021)
제5회 한국여성극작가전 - 그 집(2019)
연애플레이리스트(2018)
고형우 애니 모어 스토

In [ ]:
from collections import Counter

counter = Counter(play_actor[0])

print(play_actor)
print(counter)
print(dict(counter))

[['윤은채', '은밀하게 위대하게(2017)\n챠이카(2016)\n레베카(2016)'], ['유희준', '시간을 파는 상점(2017)'], ['하지혜', '둥근 해가 떴습니다(2012)\n연극 이(爾)(2010)\n짐(2007)'], ['백시연', '나도 이제 결혼하고 싶다(2021)\n제5회 한국여성극작가전 - 그 집(2019)\n연애플레이리스트(2018)'], ['고형우', '애니 모어 스토리(2020)\n두드려라 맥베스(2018)'], ['이인규', '수상한 흥신소(2021)\n라면(2021)\n러브 액츄얼리(2019)'], ['최정원', '컴프롬어웨이(2023)\n맘마미아 - 천안(2023)\n맘마미아 - 수원(2023)'], ['유아름', '1948 여수 - 여수(2022)\n1948 여수 - 여수(2020)\n그해 분장실(2020)'], ['신성록', '드라큘라(2023)\n벤허(2023)\n스위니토드(2022)'], ['김성진', '사명(2021)\n숙희(2021)\n언플러그드(2020)'], ['김화영', '고목(2023)\n장녀들(2023)\n스카프와 나이프(2022)'], ['이란희', '노르망디(2023)\n칸사이 주먹(2022)\n노르망디(2022)'], ['홍경인', '준생(2023)\n왓 이프(2023)\n행복(2023)'], ['홍나현', '22년 2개월(2023)\n붉은머리 안(2023)\n홍련(2023)'], ['남윤호', '코리올라누스(2021)\n보도지침(2017)\n월요쇼케이스 - 보도지침(2017)'], ['강보라', '리미트(2018)\n뷰티풀 라이프(2018)'], ['변서윤', '룸넘버13 - 울산(2018)\n룸넘버13(2016)\n내 깡패같은 로맨스(2015)'], ['김지현', '그날들 - 대구(2023)\n그날들(2023)\n스위니토드(2022)'], ['이시원', '극적인 하룻밤(2023)\n오싹한 연애(2020)\n당신 안녕(2016)'], ['엄기준', '그날들 - 수원(2023)\n그날들 - 대구(20

In [ ]:
## 뮤지컬 배우
driver.get("http://www.playdb.co.kr/artistdb/list.asp?code=013003")
time.sleep(3)
driver.find_element(By.XPATH, '//*[@id="contents"]/div[2]/table/tbody/tr[1]/td/table/tbody/tr/td[2]/a[2]').click()
time.sleep(2)
iframe_element = driver.find_element(By.ID, "list_ifm")  # iframe의 id를 사용하여 찾는 예시
driver.switch_to.frame(iframe_element)
musical_actor = []
for i in range(1, 6):
  for j in range(3, 80, 4):
    # play_DF 에 있는 장소 추출
    name = driver.find_element(By.XPATH, '/html/body/table/tbody/tr['+str(j)+']/td/table/tbody/tr/td[1]/table/tbody/tr/td[2]/a').text
    name_work = driver.find_element(By.XPATH, '/html/body/table/tbody/tr['+str(j)+']/td/table/tbody/tr/td[5]').text
    musical_actor.append([name, name_work])
    print(name, name_work)
  driver.find_element(By.XPATH, '/html/body/table/tbody/tr[84]/td/a['+str(i)+']').click()

임태경 할란카운티(2023)
지저스 크라이스트 수퍼스타(2022)
나폴레옹 헌정 콘서트(2022)
김다현 루쓰(2023)
더 그레이티스트 뮤지컬 콘서트 - 용인(2022)
미세스 다웃파이어(2022)
차지연 컴프롬어웨이(2023)
차지연 콘서트(2023)
아마데우스(2023)
김지현 그날들 - 대구(2023)
그날들(2023)
스위니토드(2022)
바다 안양 시 승격 50주년 기념 신년음악회 - 안양(2023)
제주신화월드 카운트다운 2023 - 제주(2022)
드라뮤지션 바다 콘서트 - 하남(2022)
규현 규현 & HYNN 콘서트(2023)
벤허(2023)
2023 서울파크뮤직페스티벌(2023)
박호산 기형도 플레이(2023)
오셀로(2023)
무제의 시대(2022)
주지훈 주지훈 팬미팅(2019)
생명의 항해(2010)
돈 주앙(2009)
강태을 몬테크리스토(2023)
엘리자벳(2022)
엑스칼리버(2022)
정민 스모크(2023)
사의찬미 콘서트(2023)
구텐버그(2023)
장승조 킹아더(2019)
더 데빌(2017)
구텐버그(2014)
이자람 노인과 바다 - 평택(2023)
순신(2023)
노인과 바다 - 경기 광주(2023)
김도현 배니싱(2018)
리차드3세(2018)
금강 1894 - 성남(2017)
송원근 오페라의 유령(2023)
레드북(2023)
이프덴(2022)
소냐 포르테 디 콰트로 ＆ 소냐 콘서트 - 구미(2023)
뮤지컬 갈라 콘서트 - 함양(2022)
라비앙로즈 - 화성(2022)
문혜원 타오르는 어둠 속에서(2023)
제15회 대구국제뮤지컬페스티벌 - 스페셜5(2021)
인현왕후 - 김천(2019)
이율 트레이스 유(2023)
모래시계(2022)
트레이스 유(2021)
박해미 블루 블라인드 - 의정부(2023)
박해미 콘서트 - 구리(2023)
어게인 - 여고동창생(2023)
이태원 블루 블라인드 - 의정부(2023)
헤다가블러(2023)
엘리자벳(2018)
김선경 아몬드(2022)
포미니츠(2021)
메노포즈(2018)
린아 레

In [ ]:
len(play_actor), len(musical_actor)

NameError: ignored

In [ ]:
import pandas as pd
play_actor = pd.DataFrame(play_actor)
musical_actor = pd.DataFrame(musical_actor)

In [ ]:
now = datetime.datetime.now()
play_actor.to_csv("/data/raw/play_actor" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')
musical_actor.to_csv("/data/raw/musical_actor" + now.strftime('%Y-%m-%d') + ".csv", encoding='utf-8')

## 네이버 연극정보 크롤링

In [ ]:
driver.get("https://naver.com")
time.sleep(3)
driver.find_element(By.XPATH, '//*[@id="query"]').send_keys("연극 " + play_list[1][0] + " 기본정보")
driver.find_element(By.XPATH, '//*[@id="query"]').send_keys(Keys.ENTER)
time.sleep(3)
time.sleep(2)
# 출연진 정보
driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[1]/div[3]/div/div/ul/li[3]/a').click()
time.sleep(3)
actor = []

response = requests.get(driver.current_url)
html_data = BeautifulSoup(response.text, 'html.parser')
actor_name = html_data.select('div.list_image_info > ul > li > div > div')
for a in actor_name:
  print(a.get_text())
  tag = a.get_text().split(" ")
  actor.append([tag[2], tag[5]])
string = []
for a in actor:
  string.append(", ".join(a))
act = "/".join(string)

  한혜진   사치  
  박하선   사치  
  임수향   요시노  
  서예화   요시노  
  류이재   치카  
  오한결   후타  
  이강욱   멀티  


In [ ]:
import requests
from bs4 import BeautifulSoup

driver.maximize_window()
wb = Workbook(write_only=True)
ws = wb.create_sheet('네이버 리뷰 크롤링 테스트')
ws.append(['Name', 'Genre', 'Period', 'Time', 'Place', 'Produce', 'Intro', 'Actor'])

review_site = []
# 네이버 검색창에 입력 후 엔터
for i in range(249):
  info_group = ""
  info_period = ""
  info_time = ""
  info_place = ""
  info_produce = ""
  intro = ""
  driver.get("https://naver.com")
  time.sleep(3)
  driver.implicitly_wait(1)
  driver.find_element(By.XPATH, '//*[@id="query"]').send_keys("연극 " + play_list[i][0] + " 기본정보")
  driver.find_element(By.XPATH, '//*[@id="query"]').send_keys(Keys.ENTER)
  time.sleep(3)

  try:
    # 연극 기본 정보
    info_group = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[1]/dl/div[1]/dd').text
    info_period = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[1]/dl/div[2]/dd').text
    info_time = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[1]/dl/div[3]/dd').text
    info_place = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[1]/dl/div[4]/dd').text
    info_produce = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[1]/dl/div[5]/dd').text
    intro = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[2]/div/div[2]').text
    try:
      time.sleep(2)
      # 출연진 정보
      driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[1]/div[3]/div/div/ul/li[3]/a').click()
      time.sleep(3)
      actor = []

      response = requests.get(driver.current_url)
      html_data = BeautifulSoup(response.text, 'html.parser')
      actor_name = html_data.select('div.list_image_info > ul > li > div > div')
      for a in actor_name:
        print(a.get_text())
        tag = a.get_text().split(" ")
        actor.append([tag[2], tag[5]])
      string = []
      for a in actor:
        string.append(", ".join(a))
      act = "/".join(string)
    except:
      print("출연자정보X :", i)
  except:
    print("기본정보X : ", i)


  ws.append([play_list[i][0], info_group, info_period, info_time, info_place,info_produce,intro, act])
  print(play_list[i][0], info_group, info_period, info_time, info_place,info_produce,intro, act)

wb.save('/data/raw/네이버 연극정보 크롤링 테스트_231002.xlsx')
#test1 = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[1]/div/h2/span[1]/strong/a')
#test = driver.find_element(By.CSS_SELECTOR, '#sp_nws_all1 > div.news_wrap.api_ani_send > div > a')
#print(exhibition.text)

  고영빈   민준  
  김정원   민준  
  진승욱   민준  
  피설아   수지  
  아인   수지  
  박선영   수지  
  김하늘   수지  
  윤계열   남자멀티  
  박건엽   남자멀티  
  정성화   남자멀티  
  김일강   남자멀티  
  배설하   여자멀티  
  안예슬   여자멀티  
  김예은   여자멀티  
  조하연   여자멀티  
너의목소리가들려 연극 > 코미디극 90분 2018.12.06. (목)~오픈런 월, 금 15:50, 18:00 화 ~ 목 16:00 토, 일 14:00, 16:30, 18:30 * 10/15(일), 10/22(일), 10/29(일) 14:30, 16:40 * 9/28(목), 9/30(토) ~ 10/2(월) 14:00, 16:30, 18:30 * 9/29(금) 공연 없음 * 10/3(화), 10/9(월) 14:30, 16:40 도토리씨어터 설앤수컴퍼니 재개발 지역에 발생한 갑작스러운 화재로 많은 사람들이 죽고 수지는 엄마를 잃게 된다. 방화범으로 몰린 수지는 교도소에 가게 되고... 10년의 수감생활을 마친 수지는 엄마와의 추억이 깃든 옛 동네로 발걸음을 옮긴다. 고영빈, 민준/김정원, 민준/진승욱, 민준/피설아, 수지/아인, 수지/박선영, 수지/김하늘, 수지/윤계열, 남자멀티/박건엽, 남자멀티/정성화, 남자멀티/김일강, 남자멀티/배설하, 여자멀티/안예슬, 여자멀티/김예은, 여자멀티/조하연, 여자멀티
기본정보X :  1
바닷마을다이어리 연극 > 드라마극 110분 2023.10.08. (일)~2023.11.19. (일) 화, 목, 금 19:30
수 15:00, 19:30
토 14:00, 18:00
일 16:00

* 매주 월요일 공연 없음 예술의전당 오페라하우스 자유소극장 좌석배치도 라이브러리컴퍼니  고영빈, 민준/김정원, 민준/진승욱, 민준/피설아, 수지/아인, 수지/박선영, 수지/김하늘, 수지/윤계열, 남자멀티/박건엽, 남자멀티/정성화, 남자멀티/김일강, 남자멀티/배설하, 여자멀티/안예슬, 여

In [ ]:
wb.save('/data/raw/네이버 연극정보 크롤링 테스트_231001.xlsx')

In [ ]:
wb = Workbook(write_only=True)
ws = wb.create_sheet('네이버 리뷰 크롤링 테스트')
ws.append(['Name', 'ID', 'Score', 'Review'])

for j in range(30):
  # 링크로 이동
  driver.get(exhib_review_link_2['link'][j])
  time.sleep(2)
  test1 = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[1]/h3/span[2]/em/span')
  print(test1.text)
  # 리뷰 5개씩 크롤링 # 이걸 여러개 ...
  try:
    for t in range(100): # 최대 500개씩
      for i in range(1, 6):
        name = exhib_review_link_2['전시회 이름'][j]
        id = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[1]/a[2]/span[1]').text
        score = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[3]/div/span').text
        review = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/p/span').text
        ws.append([name, id, score, review])
        print(name, id, score, review)
      driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[2]/a[2]').click()
      time.sleep(3)
  except:
    continue

#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[1]/review-list-item/div/div
#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[2]/review-list-item/div/div

wb.save('/data/raw/네이버 리뷰 크롤링 테스트_230925.xlsx')

## 네이버 리뷰 크롤링

In [ ]:
exhib_list = exhib_string.values.tolist()
exhib_list

[['거장의 시선, 사람을 향하다 - 영국 내셔널갤러리 명화전'],
 [' 일리야 밀스타인'],
 ['이경준 사진전: 원 스텝 어웨이'],
 ['백희나 그림책전'],
 ['2023년 9,10월-어둠속의대화'],
 ['럭스:시적해상도'],
 ['에르베 튈레展'],
 ['모네 인사이드'],
 ['스티키몬스터랩: 스틸 라이프'],
 ['스폰지밥의 우당탕탕 시간여행展'],
 ['요시고 사진전 - 부산'],
 ['진격의 거인전 FINAL in SEOUL'],
 [' 반고흐 인 서울'],
 ['문도 멘도: 판타스틱 시티 라이프'],
 ['스즈메의 문단속 展 in SHINSEGAE'],
 [' 오스틴 리: 패싱타임'],
 ['인사이드미 : 잃어버린 나의 감정을 찾아서'],
 ['2023년 9,10월-어둠속의대화'],
 ['알폰스무하 이모션 인 서울'],
 ['세르주 블로크展 : KISS'],
 ['동물 없는 동물원 展'],
 ['CAT ART : 고양이 미술사'],
 ['알폰스 무하: 더 골든 에이지'],
 ['빈센트 발 : ART OF SHADOW'],
 ['에곤실레와 클림트'],
 [' 창경궁 야경투어'],
 ['에바 알머슨 특별전 : 에바 알머슨, Andando'],
 ['팀보타 누적관람객 300만 - 탐화림 : 화림이 품은 탐의 이야기'],
 ['덕수궁 야경투어'],
 [' 어린왕자 인 서울'],
 ['2023국제농업박람회 - 순천'],
 [' 두근두근 도라에몽展'],
 ['맥스달튼 전시전 EP.2 영화의 순간들'],
 ['제22회 서울카페쇼 2023'],
 ['극장판 명탐정 코난: 흑철의 어영'],
 ['극장판 명탐정 코난: 흑철의 어영'],
 [' 반고흐 인 서울'],
 ['스즈메의 문단속 展 in SHINSEGAE'],
 [' 로렌 차일드 : 요정처럼 생각하기 展'],
 [' 2023 울산국제아트페어'],
 ['진격의 거인전 FINAL in SEOUL'],
 ['진격의 거인전 FINAL in SEOUL'],
 [' 빛의 시어터 입장권'],
 [' 아르떼뮤지엄'

In [ ]:
driver.maximize_window()

review_site = []
# 네이버 검색창에 입력 후 엔터
for i in range(153):
  driver.get("https://naver.com")
  time.sleep(3)
  driver.implicitly_wait(1)
  driver.find_element(By.XPATH, '//*[@id="query"]').send_keys(exhib_string[i])
  driver.find_element(By.XPATH, '//*[@id="query"]').send_keys(Keys.ENTER)
  time.sleep(3)


  # 전시 정보창이 있는지 확인하기 위함
  #exhibition = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]')
  #time.sleep(3)

  # 있다면 ...
  try:
    # 전시 정보 더보기로 이동
    driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div/div/div[3]/a/div').click()
    time.sleep(2)
    # 전시 예매로 이동
    driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[2]/div[1]/div/div[3]/div/ul/li[1]/a').click()
    time.sleep(2)
    # 예매자 리뷰 더보기 클릭
    driver.find_element(By.CSS_SELECTOR, '#container > div > div > div.wrap_review_lst > a > span').click()
    time.sleep(3)
    print(driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[1]/h3/span[1]').text)
    # 리뷰 사이트 append
    review_site.append(driver.current_url)
  except:
    review_site.append('')



#test1 = driver.find_element(By.XPATH, '//*[@id="main_pack"]/div[2]/div[1]/div/h2/span[1]/strong/a')
#test = driver.find_element(By.CSS_SELECTOR, '#sp_nws_all1 > div.news_wrap.api_ani_send > div > a')
#print(exhibition.text)
print(review_site)

예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
예매자 리뷰
['https://booking.naver.com/review/bizes/899755', '', '', '', '', '', '', '', 'https://booking.naver.com/review/bizes/949610', 'https://booking.naver.com/review/bizes/955615', '', '', '', 'https://booking.naver.com/review/bizes/875642', '', '', '', '', '', '', 'https://booking.naver.com/review/bizes/968616', '', 'https://booking.naver.com/review/bizes/974375', 'https://booking.naver.com/review/bizes/958812', '', '', '', '', '', '', '', 'https://booking.naver.com/review/bizes/958915', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'https://booking.naver.com/review/bizes/882663', '', '', '', '', '', '', '', 'https://booking.naver.com/review/bizes/837936', ''

In [ ]:
wb = Workbook(write_only=True)
ws = wb.create_sheet('네이버 리뷰 크롤링 테스트')
ws.append(['Name', 'ID', 'Score', 'Review'])

for j in range(30):
  # 링크로 이동
  driver.get(exhib_review_link_2['link'][j])
  time.sleep(2)
  test1 = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[1]/h3/span[2]/em/span')
  print(test1.text)
  # 리뷰 5개씩 크롤링 # 이걸 여러개 ...
  try:
    for t in range(100): # 최대 500개씩
      for i in range(1, 6):
        name = exhib_review_link_2['전시회 이름'][j]
        id = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[1]/a[2]/span[1]').text
        score = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[3]/div/span').text
        review = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/p/span').text
        ws.append([name, id, score, review])
        print(name, id, score, review)
      driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[2]/a[2]').click()
      time.sleep(3)
  except:
    continue

#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[1]/review-list-item/div/div
#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[2]/review-list-item/div/div

wb.save('/data/raw/네이버 리뷰 크롤링 테스트_230925.xlsx')

In [ ]:
review_site

['https://booking.naver.com/review/bizes/899755',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'https://booking.naver.com/review/bizes/949610',
 'https://booking.naver.com/review/bizes/955615',
 '',
 '',
 '',
 'https://booking.naver.com/review/bizes/875642',
 '',
 '',
 '',
 '',
 '',
 '',
 'https://booking.naver.com/review/bizes/968616',
 '',
 'https://booking.naver.com/review/bizes/974375',
 'https://booking.naver.com/review/bizes/958812',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'https://booking.naver.com/review/bizes/958915',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'https://booking.naver.com/review/bizes/882663',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'https://booking.naver.com/re

In [ ]:
exhib_rank['link'] = review_site

In [ ]:
exhib_rank.to_csv('/data/raw/크롤링 리뷰 링크.csv', index=None)

## 리뷰 링크 데이터 불러오기

In [ ]:
import pandas as pd
exhib_rank = pd.read_csv('/data/raw/크롤링 리뷰 링크_230925.csv', encoding='utf-8')
exhib_rank = pd.DataFrame(exhib_rank)
exhib_rank

,전시회 이름,link,Unnamed: 2
0,"거장의 시선, 사람을 향하다 - 영국 내셔널갤러리 명화전",https://booking.naver.com/review/bizes/899755,NaN
1,［얼리버드］ 일리야 밀스타인,https://booking.naver.com/review/bizes/963297,NaN
2,［얼리버드］이경준 사진전: 원 스텝 어웨이,NaN,NaN
3,백희나 그림책전,https://booking.naver.com/review/bizes/923312,NaN
4,"［북촌］2023년 9,10월-어둠속의대화(DIALOGUE IN THE DARK)",NaN,NaN
...,...,...,...
148,한국만화박물관 입장권,NaN,NaN
149,헤이리 파주공룡박물관,NaN,NaN
150,후지시로 세이지 북촌스페이스,NaN,NaN
151,ART ROAD SHOWCASE ＆ ART EXHIBITION in KOREA 20...,NaN,NaN


In [ ]:
exhib_review_link = exhib_rank[['전시회 이름', 'link']]
exhib_review_link

,전시회 이름,link
0,"거장의 시선, 사람을 향하다 - 영국 내셔널갤러리 명화전",https://booking.naver.com/review/bizes/899755
1,［얼리버드］ 일리야 밀스타인,https://booking.naver.com/review/bizes/963297
2,［얼리버드］이경준 사진전: 원 스텝 어웨이,NaN
3,백희나 그림책전,https://booking.naver.com/review/bizes/923312
4,"［북촌］2023년 9,10월-어둠속의대화(DIALOGUE IN THE DARK)",NaN
...,...,...
148,한국만화박물관 입장권,NaN
149,헤이리 파주공룡박물관,NaN
150,후지시로 세이지 북촌스페이스,NaN
151,ART ROAD SHOWCASE ＆ ART EXHIBITION in KOREA 20...,NaN


In [ ]:
pd.isnull(exhib_review_link).sum()

전시회 이름      0
link      123
dtype: int64

In [ ]:
exhib_review_link_2 = exhib_review_link
exhib_review_link_2.dropna(axis=0)
exhib_review_link_2 = exhib_review_link_2.reset_index(drop=True)

In [ ]:
exhib_review_link_2.count()

전시회 이름    30
link      30
dtype: int64

In [ ]:
exhib_review_link_2

,전시회 이름,link
0,"거장의 시선, 사람을 향하다 - 영국 내셔널갤러리 명화전",https://booking.naver.com/review/bizes/899755
1,［얼리버드］ 일리야 밀스타인,https://booking.naver.com/review/bizes/963297
2,백희나 그림책전,https://booking.naver.com/review/bizes/923312
3,［9월예매］모네 인사이드,https://booking.naver.com/review/bizes/974352
4,스티키몬스터랩: 스틸 라이프,https://booking.naver.com/review/bizes/949610
5,스폰지밥의 우당탕탕 시간여행展,https://booking.naver.com/review/bizes/955615
6,요시고 사진전 - 부산,https://booking.naver.com/review/bizes/885136
7,진격의 거인전 FINAL in SEOUL,https://booking.naver.com/review/bizes/512380
8,문도 멘도: 판타스틱 시티 라이프,https://booking.naver.com/review/bizes/875642
9,알폰스무하 이모션 인 서울,https://booking.naver.com/review/bizes/944048


In [ ]:
exhib_review_link_2['link'][0]

'https://booking.naver.com/review/bizes/899755'

In [ ]:
wb = Workbook(write_only=True)
ws = wb.create_sheet('네이버 리뷰 크롤링 테스트')
ws.append(['Name', 'ID', 'Score', 'Review'])

for j in range(30):
  # 링크로 이동
  driver.get(exhib_review_link_2['link'][j])
  time.sleep(2)
  test1 = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[1]/h3/span[2]/em/span')
  print(test1.text)
  # 리뷰 5개씩 크롤링 # 이걸 여러개 ...
  try:
    for t in range(100): # 최대 500개씩
      for i in range(1, 6):
        name = exhib_review_link_2['전시회 이름'][j]
        id = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[1]/a[2]/span[1]').text
        score = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/div[3]/div/span').text
        review = driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[' + str(i) + ']/review-list-item/div/div/p/span').text
        ws.append([name, id, score, review])
        print(name, id, score, review)
      driver.find_element(By.XPATH, '//*[@id="container"]/review-list/div/div/div/div[2]/a[2]').click()
      time.sleep(3)
  except:
    continue

#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[1]/review-list-item/div/div
#//*[@id="container"]/review-list/div/div/div/div[1]/div/div[3]/ul/li[2]/review-list-item/div/div

wb.save('/data/raw/네이버 리뷰 크롤링 테스트_230925.xlsx')

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
"이 그림 본적 있어"라며 아는 척도 할 수 있을거 같습니다. 이렇게 작품을 아이들에게 노출 시켜주는것만으로도 공부가 될수 있네요
아는 엄마들에게 열심히 홍보했습니다
즐거운 추억이었습니다
［9월예매］모네 인사이드 chy**** 5 미디어 아트여서 그런거 모네, 르누아르, 세잔 등 일대기를 볼 수 있었던 좋은 전시가 된것 같아요 백화점 안에 있어서 전시보고 백화점 구경도 하고 편하고 재미있게 다녀온것 같습니당 ✌🏻
［9월예매］모네 인사이드 타미8 4 화가의 일대기를 작품과 매치시켜 이야기를 하듯 설명을 곁들여 화가의 작품세계에 쉽게 접근할 수 있었고 이해하기도 좋은 시간이였습니다. 모네의 마지막작품을 원화로 보고싶어 프랑스 오랑주리에 가고싶다는 생각이 들었습니다.
［9월예매］모네 인사이드 soar48 4 미디어 아트로 모네를 만날 수 있어 더욱 실감나고 황홀합니다.
［9월예매］모네 인사이드 mei**** 3 
［9월예매］모네 인사이드 peacemaker2227 5 믿고 가는 그라운드시소입니다^^ 흥미진진하게 미술 교양 수업 듣고 나온 기분이예요.
［9월예매］모네 인사이드 green9 4 모네를 좋아하기 때문에 그리고 미디어아트도 좋아해서 좋은 전시회 보고 왔습니다.
전시회라기 보다는 상영이라고 하는 것이 맞겠지요.
편안하세 앉아서 멋진 전시회 감상 잘했어요.
［9월예매］모네 인사이드 따가운뒤통수 5 
［9월예매］모네 인사이드 mei**** 1.5 35분 영상에 포토타임 20분 정도인데.. 전시개념보다는 미디어아트라 보면 맞고..
모네인사이드가 알폰스무하 더 골든에이지보다 실속이 없었다
모네만 보고 9900원 낼 바에야 15000원으로 무하전까지 보고 나오는 게 돈이 덜 아까울 듯
［9월예매］모네 인사이드 초코4178 5 모네 그림들 너무 이쁨!!
［9월예매］모네 인사이드 shara0225 4.5 아주 좋았어요
［9월예매］모네 인사이드 큐앤에이 4 작품을 영상으로 접하니 더 생생하고 좋았어요.
모네의

In [ ]:
# print(rows[:2])
text = []
for i, row in enumerate(rows):
    # if i >2 : break
    #print(row.text)
    text.append(row.text)
#text/html/body/table/tbody/tr[2]/td[3]/div/div/div[2]/div/table/tbody/tr[1]/td[2]/span
#body > table > tbody > tr:nth-child(2) > td:nth-child(3) > div > div > div.con > div > table > tbody
#body > table > tbody > tr:nth-child(2) > td:nth-child(3) > div > div > div.con > div > table > tbody > tr:nth-child(1) > td.RKtxt > span
#body > table > tbody > tr:nth-child(2) > td:nth-child(3) > div > div > div.con > div > table > tbody > tr:nth-child(153)